# Atividade: Assistente de IA Generativa com Hugging Face e Gemini

**Disciplina:** Disruptive Architectures: IoT, IoB & Generative AI

**Grupo:**
- Nicholas Albuquerque Buzo (RM561082)
- Gustavo Gomes (RM555999)
- Matheus Vecchi (RM)

**Tema escolhido:** (escreva aqui o tema da lista do README)

---

Neste notebook o grupo vai construir um assistente de IA em 4 etapas (e 1 bônus):

1. Assistente com personalidade (Hugging Face)
2. Comparação Hugging Face x Gemini
3. Chat com memória
4. Interface web com Gradio
5. (Bônus) API com FastAPI

Os trechos marcados com **`# >>> PERSONALIZE`** devem ser alterados pelo grupo.
Execute as células em ordem, de cima para baixo.

## 0. Configuração

In [1]:
# huggingface_hub -> cliente para chamar modelos remotamente (API do Hugging Face)
# google-genai    -> SDK oficial do Google Gemini
# gradio          -> cria interfaces web a partir de funções Python
!pip install huggingface_hub google-genai gradio -q

In [3]:
from huggingface_hub import InferenceClient
from google import genai
from google.genai import types
from google.colab import userdata

# Os tokens ficam nos Secrets do Colab (ícone de chave na barra lateral)
HF_TOKEN = userdata.get("HF_TOKEN")
GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")

print("HF_TOKEN ok" if HF_TOKEN else "ERRO: adicione HF_TOKEN nos Secrets do Colab")
print("GEMINI_API_KEY ok" if GEMINI_API_KEY else "ERRO: adicione GEMINI_API_KEY nos Secrets do Colab")

HF_TOKEN ok
GEMINI_API_KEY ok


In [21]:
# Modelos usados na atividade (os mesmos da Aula 05)
MODELO_HF = "meta-llama/Llama-3.1-8B-Instruct"
MODELO_GEMINI = "gemini-3.5-flash-lite"

cliente_hf = InferenceClient(model=MODELO_HF, token=HF_TOKEN, provider="auto")
cliente_gemini = genai.Client(api_key=GEMINI_API_KEY)

In [22]:
# >>> PERSONALIZE: descreva o assistente do grupo de acordo com o tema escolhido.
# Este é o "system": a instrução que define o comportamento do assistente.
# O exemplo abaixo é do tema 1 (casa inteligente). Troque pelo tema do grupo.

SYSTEM_PROMPT = """Você é um consultor de agricultura inteligente.
Ajude o usuário com dúvidas sobre irrigação, umidade do solo e estações meteorológicas conectadas.
Responda sempre em português, de forma clara, em no máximo 5 frases.
Se a pergunta não tiver relação com agricultura inteligente, diga educadamente que não pode ajudar."""

---
## Etapa 1: Assistente com personalidade (Hugging Face)

Cada mensagem enviada ao modelo tem uma etiqueta `role` que diz quem escreveu:

- `system`: a regra que o assistente deve seguir
- `user`: a pergunta do usuário

Nesta etapa, a **mesma pergunta** é enviada três vezes, e **só o `system` muda**:

| Chamada | system | user |
|---|---|---|
| 1 | Especialista no tema do grupo | mesma pergunta |
| 2 | Professor para crianças | mesma pergunta |
| 3 | Resposta em uma frase | mesma pergunta |

Se as respostas saírem diferentes, a diferença veio só do `system`. É assim que se criam assistentes diferentes em cima do mesmo modelo.

In [23]:
def perguntar_hf(pergunta, system, temperatura=0.7):
    """Envia uma pergunta ao modelo do Hugging Face e devolve o texto da resposta."""
    mensagens = [
        {"role": "system", "content": system},   # como o assistente deve se comportar
        {"role": "user",   "content": pergunta}, # o que o usuário perguntou
    ]
    resposta = cliente_hf.chat_completion(
        messages=mensagens,
        max_tokens=300,
        temperature=temperatura,
    )
    return resposta.choices[0].message.content

In [24]:
# >>> PERSONALIZE: crie 3 personalidades diferentes para o assistente do grupo.
personalidades = {
    "Especialista": SYSTEM_PROMPT,
    "Consultor de lavouras": "Você tem respostas específicas para as lavouras e plantações, cuidados e prevenções dadas as perguntas do usuário.",
    "Técnico": "Você responde as perguntas com maior teor técnico, ensinando o passo a passo e teorias para o usuário.",
}

# >>> PERSONALIZE: uma pergunta relacionada ao tema do grupo.
pergunta = "Possuo uma plantação de café plantada no início do ano, desejo aumentar a safra e também a qualidade do plantio, pode me ajudar?"

for nome, system in personalidades.items():
    print(f"===== {nome} =====")
    print(perguntar_hf(pergunta, system))
    print()

===== Especialista =====
Claro, posso ajudar! Para aumentar a safra e a qualidade do plantio de café, é importante monitorar a umidade do solo e o nível de irrigação. Uma estação meteorológica conectada pode ser uma ferramenta valiosa para isso, pois pode fornecer dados precisos sobre a umidade do solo, a temperatura e a precipitação. Além disso, é importante escolher a variedade de café mais adequada para a região e cuidar da fertilização e da prática de rotação de culturas. Vamos discutir os detalhes!

===== Consultor de lavouras =====
Claro, posso te ajudar!

Para aumentar a safra e a qualidade do seu plantio de café, aqui estão algumas dicas específicas:

**Aumentar a Safra:**

1.  **Adubação:** O café é uma cultura que precisa de nutrientes para crescer e produzir frutos. Verifique se a adubação está em dia e considere adicionar fertilizantes orgânicos ou inorgânicos para melhorar a fertilidade do solo.
2.  **Cuidado com as pragas:** Pragas como a broca do café podem danificar as 

**Observações do grupo (Etapa 1):**

- O que mudou nas respostas de cada personalidade?
- Qual `system` gerou a resposta mais útil para o tema? Por quê?

O "especialista" utilizou uma resposta única e tentou continuar o contato com o usuário retornando uma pergunta, os outros dois geraram uma resposta mais parecida, talvez devido a pergunta, mas separaram a resposta em tópicos com dicas e passo a passo. Prefiro a resposta do primeiro por tentar manter a comunicação com o usuário

---
## Etapa 2: Hugging Face x Gemini

Agora as mesmas perguntas vão para **dois modelos diferentes**.

No Gemini, o `system` não vai dentro da lista de mensagens. Ele é passado no parâmetro `system_instruction`.

In [25]:
def perguntar_gemini(pergunta, system):
    """Envia uma pergunta ao Gemini e devolve o texto da resposta."""
    resposta = cliente_gemini.models.generate_content(
        model=MODELO_GEMINI,
        contents=pergunta,
        config=types.GenerateContentConfig(
            system_instruction=system,  # equivalente ao role "system"
            max_output_tokens=300,
        ),
    )
    return resposta.text

In [26]:
# >>> PERSONALIZE: 3 perguntas sobre o tema do grupo.
perguntas = [
    "Quais sensores são mais usados em uma plantação de café?",
    "Qual a melhor época para plantio e colheita do café?",
    "Quais cuidados devo ter com o clima e irrigação?",
]

for p in perguntas:
    print("PERGUNTA:", p)
    print("\n--- Hugging Face (Llama) ---")
    print(perguntar_hf(p, SYSTEM_PROMPT))
    print("\n--- Gemini ---")
    print(perguntar_gemini(p, SYSTEM_PROMPT))
    print("\n" + "=" * 60 + "\n")

PERGUNTA: Quais sensores são mais usados em uma plantação de café?

--- Hugging Face (Llama) ---
Em uma plantação de café, os sensores mais comuns utilizados são:

*   Sensores de temperatura e umidade do solo para monitorar as condições do solo e evitar doenças;
*   Sensores de irrigação para controlar a quantidade de água aplicada e evitar desperdício;
*   Sensores de radiação solar para monitorar a exposição ao sol e ajustar a irrigação;
*   Sensores de pH do solo para monitorar a acidez do solo e ajustar a fertilização.
Esses sensores ajudam a otimizar a irrigação e reduzir o consumo de água e fertilizantes.

--- Gemini ---
Na cultura do café, os sensores mais utilizados são os de umidade do solo, fundamentais para monitorar a retenção de água e definir o momento exato da irrigação. Também são muito empregados os sensores de temperatura e umidade relativa do ar, essenciais para prevenir doenças fúngicas. Anemômetros e pluviômetros, integrados às estações meteorológicas, ajudam a re

**Observações do grupo (Etapa 2):**

- Os dois modelos seguiram as regras do `SYSTEM_PROMPT` (idioma, tamanho, tema)?
- Qual respondeu melhor? Em qual pergunta a diferença foi maior?

Os dois modelos respoderam bem, porém o Gemini foi mais consistente. Gostei mais dar respostas do Gemini na segunda e terceira pergunta, porém o llama performou melhor na primeira e inclusive retornou uma boa definição com tópicos.

---
## Etapa 3: Chat com memória

O modelo **não guarda memória** entre uma chamada e outra.
Quem guarda a conversa é o nosso código, na lista `historico`, que é enviada inteira a cada mensagem.

Comandos do chat:
- `sair` encerra o chat
- `limpar` apaga o histórico (o assistente "esquece" a conversa)
- `historico` mostra quantas mensagens estão guardadas

**Teste sugerido:** diga seu nome, pergunte "qual é o meu nome?", digite `limpar` e pergunte de novo.

In [29]:
historico = [{"role": "system", "content": SYSTEM_PROMPT}]

print("Chat iniciado! Comandos: sair | limpar | historico\n")

while True:
    entrada = input("Você: ")

    if entrada.lower() == "sair":
        print("Encerrando chat.")
        break

    if entrada.lower() == "limpar":
        historico = [{"role": "system", "content": SYSTEM_PROMPT}]  # mantém só o system
        print("\n(histórico apagado)\n")
        continue

    if entrada.lower() == "historico":
        print(f"\n(mensagens no histórico: {len(historico)})\n")
        continue

    historico.append({"role": "user", "content": entrada})

    resposta = cliente_hf.chat_completion(messages=historico, max_tokens=300)
    texto = resposta.choices[0].message.content

    historico.append({"role": "assistant", "content": texto})

    print(f"\nAssistente: {texto}\n")

Chat iniciado! Comandos: sair | limpar | historico

Você: meu nome é pedro

Assistente: Olá Pedro! Estou aqui para ajudar com questões sobre agricultura inteligente. Qual é o seu problema ou dúvida? Quer saber sobre irrigação, umidade do solo ou estações meteorológicas conectadas?

Você: qual é o meu nome?

Assistente: Seu nome é Pedro! Estou aqui para ajudar você com questões sobre agricultura inteligente. Qual é a sua dúvida específica?

Você: limpar

(histórico apagado)

Você: qual o meu nome?

Assistente: Peço desculpas, mas não tenho acesso a informações sobre você ou seu nome. Posso ajudá-lo com questões relacionadas à agricultura inteligente, como irrigação, umidade do solo e estações meteorológicas conectadas?

Você: meu nome é pedro!

Assistente: Muito bem, Pedro! Agora que sabemos o seu nome, podemos continuar. O que gostaria de saber sobre irrigação, umidade do solo ou estações meteorológicas conectadas? Estou aqui para ajudar!

Você: historico

(mensagens no histórico: 5)



In [30]:
# Veja como ficou a lista enviada ao modelo
for msg in historico:
    print(f"[{msg['role']}] {msg['content'][:80].replace(chr(10), ' ')}")

[system] Você é um consultor de agricultura inteligente. Ajude o usuário com dúvidas sobr
[user] qual o meu nome?
[assistant] Peço desculpas, mas não tenho acesso a informações sobre você ou seu nome. Posso
[user] meu nome é pedro!
[assistant] Muito bem, Pedro! Agora que sabemos o seu nome, podemos continuar. O que gostari


**Observações do grupo (Etapa 3):**

- O que aconteceu quando vocês perguntaram o nome antes e depois do `limpar`?
- Explique, com suas palavras, por que isso acontece.

Antes de limpar a IA consegue retornar o nome digitado, depois do limpar já não possui acesso, e também não é encontrada ao dar o print no histórico

---
## Etapa 4: Interface web com Gradio

O assistente ganha uma interface web, com a opção de escolher o modelo (Hugging Face ou Gemini).

O Gradio entrega o histórico da conversa já no formato de `role`/`content`.
A função `texto_da_mensagem` existe porque, dependendo da versão do Gradio, o conteúdo chega como texto simples ou como lista.

In [31]:
import gradio as gr

def texto_da_mensagem(conteudo):
    """Extrai o texto de uma mensagem do histórico do Gradio."""
    if isinstance(conteudo, str):
        return conteudo
    if isinstance(conteudo, list):
        return " ".join(item.get("text", "") for item in conteudo if isinstance(item, dict))
    return str(conteudo)


def responder(mensagem, historico_gradio, provedor):
    if provedor == "Hugging Face":
        # Formato HF: roles system, user e assistant
        mensagens = [{"role": "system", "content": SYSTEM_PROMPT}]
        for msg in historico_gradio:
            mensagens.append({"role": msg["role"], "content": texto_da_mensagem(msg["content"])})
        mensagens.append({"role": "user", "content": mensagem})

        resposta = cliente_hf.chat_completion(messages=mensagens, max_tokens=300)
        return resposta.choices[0].message.content

    else:
        # Formato Gemini: roles user e model; o system vai em system_instruction
        conteudos = []
        for msg in historico_gradio:
            papel = "model" if msg["role"] == "assistant" else "user"
            conteudos.append({"role": papel, "parts": [{"text": texto_da_mensagem(msg["content"])}]})
        conteudos.append({"role": "user", "parts": [{"text": mensagem}]})

        resposta = cliente_gemini.models.generate_content(
            model=MODELO_GEMINI,
            contents=conteudos,
            config=types.GenerateContentConfig(system_instruction=SYSTEM_PROMPT, max_output_tokens=300),
        )
        return resposta.text

In [32]:
# >>> PERSONALIZE: título, descrição e exemplos de acordo com o tema do grupo.
gr.ChatInterface(
    fn=responder,
    additional_inputs=[gr.Radio(["Hugging Face", "Gemini"], value="Hugging Face", label="Modelo")],
    title="Consultor de agricultura inteligente",
    description="Pergunte sobre automação agrícola, sensores e cuidados tecnológicos com a safra.",
    examples=[
        ["Como funciona sensores para plantação?", "Hugging Face"],
        ["Vale a pena métodos de automatização para irrigação?", "Gemini"],
    ],
).launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://ee9852488828e081b0.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


**Observações do grupo (Etapa 4):**

- Tire um print da interface funcionando e coloque no README do repositório do grupo.
- Troque de modelo no meio da conversa. O assistente continuou lembrando do que foi dito? Por quê?

Não foi armazenado o contexto da conversa na troca de modelo, o assistente informou que não armazena histórico.

> Para parar a interface, interrompa a célula (botão de parar do Colab).

---
## Etapa 5 (Bônus): API com FastAPI

Aqui o assistente vira uma **API REST**, como no final da Aula 05.
Qualquer frontend (site, app, dispositivo IoT) poderia chamar esse endpoint.

A célula abaixo cria o arquivo `app.py`.

In [33]:
%%writefile app.py
import os
from fastapi import FastAPI
from pydantic import BaseModel
from huggingface_hub import InferenceClient

# O token e o system prompt vêm de variáveis de ambiente (nunca escreva o token no código)
client = InferenceClient(
    model="meta-llama/Llama-3.1-8B-Instruct",
    token=os.environ["HF_TOKEN"],
    provider="auto",
)
SYSTEM_PROMPT = os.environ.get("SYSTEM_PROMPT", "Você é um assistente prestativo.")

app = FastAPI()

class Pergunta(BaseModel):
    mensagem: str

@app.post("/chat")
def chat(pergunta: Pergunta):
    resposta = client.chat_completion(
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": pergunta.mensagem},
        ],
        max_tokens=300,
    )
    return {"resposta": resposta.choices[0].message.content}

Writing app.py


In [34]:
# Inicia o servidor em segundo plano, dentro do próprio Colab
!pip install fastapi uvicorn -q

import os, subprocess, time
os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["SYSTEM_PROMPT"] = SYSTEM_PROMPT

servidor = subprocess.Popen(["uvicorn", "app:app", "--port", "8000"])
time.sleep(5)  # espera o servidor subir
print("Servidor rodando em http://localhost:8000")

Servidor rodando em http://localhost:8000


In [36]:
# Testa o endpoint como um frontend faria (HTTP POST com JSON)
import requests

r = requests.post("http://localhost:8000/chat", json={"mensagem": "O que é Automação agrícola?"})
print(r.status_code)
print(r.json()["resposta"])

200
A Automação agrícola é um conjunto de tecnologias e sistemas que visam melhorar a eficiência e produtividade da agricultura, utilizando sensores, sistemas de informação e controle remoto. Isso inclui a irrigação automática, o monitoramento da umidade do solo, e a coleta de dados com estações meteorológicas conectadas.

A Automação agrícola pode ajudar a reduzir o desperdício de recursos, aumentar a produtividade e melhorar a qualidade dos produtos agrícolas.

Se você tiver mais alguma dúvida, sinta-se à vontade para perguntar!


In [37]:
# Encerra o servidor
servidor.terminate()
print("Servidor encerrado.")

Servidor encerrado.
